In [ ]:
import sys
# sys.path.insert(0, '.')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torch.utils.tensorboard import SummaryWriter
from torchmetrics.classification import MulticlassF1Score
from PIL import Image
import os
import pandas as pd
from tqdm import tqdm
from datetime import datetime
from torchvision import transforms
from src.model import build_vit
from src.augmentations import AUGMENTATIONS #, get_train_transform, val_transform
from torch.utils.tensorboard import SummaryWriter
from torchmetrics.classification import MulticlassF1Score, MulticlassAccuracy, MulticlassPrecision, MulticlassRecall

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

# Dataset

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, csv_path, split='train', transform=None):

        self.transform = transform
        
        full_df = pd.read_csv(csv_path)

        self.classes = sorted(full_df['label'].unique())
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        
        self.df = full_df[full_df['split'] == split].reset_index(drop=True)
        
        if len(self.df) == 0:
            raise ValueError(f"No samples found for split: {split}. Check your CSV or directory paths.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['abs_path']
        label_name = row['label']
        
        label = self.class_to_idx[label_name]

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            raise e

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
image_size = 96
base_pipeline = [
    transforms.Resize((image_size, image_size)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]

csv_path = '../data/dataset.csv'

test_dataset = ImageDataset(csv_path=csv_path, split='test', transform=transforms.Compose(base_pipeline))
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
num_classes=4
f1_metric = MulticlassF1Score(num_classes=num_classes, average=None).to(device)
macro_f1_metric = MulticlassF1Score(num_classes=num_classes, average='macro').to(device)
accuracy_metric = MulticlassAccuracy(num_classes=num_classes, average='macro').to(device)
precision_metric = MulticlassPrecision(num_classes=num_classes, average='macro').to(device)
recall_metric = MulticlassRecall(num_classes=num_classes, average='macro').to(device)
criterion = nn.CrossEntropyLoss()

# Testing

In [ ]:
results = []

for metric in [f1_metric, macro_f1_metric, accuracy_metric, precision_metric, recall_metric]:
    metric.to(device)

print(f"\n{'='*30}\nStarting Test Set Evaluation\n{'='*30}")

model_names = list(AUGMENTATIONS.keys())
print(model_names)
# model_names = ["baseline"]

for aug_name in model_names:
    description = f"vit_model_96x96_{aug_name}"
    model_path = f'./models_only_baseline_100/best_{description}_1.pth'
    
    print(f"Evaluating: {aug_name}...")

    model = build_vit(num_classes=num_classes, img_size=image_size).to(device)
    
    try:
        model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    except FileNotFoundError:
        print(f"Skipping {aug_name}: {model_path} not found.")
        continue

    model.eval()

    f1_metric.reset()
    macro_f1_metric.reset()
    accuracy_metric.reset()
    precision_metric.reset()
    recall_metric.reset()

    with torch.no_grad():
        for img, labels in test_loader:
            img, labels = img.to(device), labels.to(device)
            output = model(img)

            f1_metric.update(output, labels)
            macro_f1_metric.update(output, labels)
            accuracy_metric.update(output, labels)
            precision_metric.update(output, labels)
            recall_metric.update(output, labels)

    row = {
        "Augmentation": aug_name,
        "Accuracy": accuracy_metric.compute().item(),
        "Macro_F1": macro_f1_metric.compute().item(),
        "Precision": precision_metric.compute().item(),
        "Recall": recall_metric.compute().item()
    }
    
    per_class_f1 = f1_metric.compute().tolist()
    for i, score in enumerate(per_class_f1):
        row[f"Class_{i}_F1"] = score

    results.append(row)

    del model
    torch.cuda.empty_cache()

In [ ]:
pd.DataFrame(results)

In [ ]:
# baseline - same batch 64, same lr 1e3, Changed normalisation (pre+aug+post)
# rest data aug - batch 32, same lr 1e4, but same old normalisation (pre + imageNet norm + aug)
# 
# Augmentation	Accuracy	Macro_F1	Precision	Recall	Class_0_F1	Class_1_F1	Class_2_F1	Class_3_F1
# 0	baseline	0.890465	0.889934	0.904234	0.890465	0.824114	0.866667	0.906155	0.962801


# Augmentation	Accuracy	Macro_F1	Precision	Recall	Class_0_F1	Class_1_F1	Class_2_F1	Class_3_F1
# 0	baseline	0.91143	0.90575	0.902109	0.91143	0.868085	0.891473	0.901515	0.961926
# -----
# 50% aug only baseline
# 
# Augmentation	Accuracy	Macro_F1	Precision	Recall	Class_0_F1	Class_1_F1	Class_2_F1	Class_3_F1
# 0	baseline	0.945871	0.943286	0.94666	0.945871	0.890071	0.931397	0.966469	0.985205


# 100% aug only baseline